## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Kevin Teng

**Date:** 10/2/2026

Q1.1)

Classifications sort observations into specific categories. The category-based outcomes ensure that the observation is assigned to one of the possible labels. In contrast, regressions are continuous and predict a value on that continuous range rather than assigning an observation to a specific category.


Q1.2)

A confusion table is a grid/table that shows the performance of a model by comparing the predicted value to the actual value. It shows whether the predicted value is correct or misclassified. Confusion tables allow us to understand the overall accuracy of the model and which specific class is better at predicting correctly.


Q1.3)

SSE (Sum of Squared Errors) is calculated by summing the squares of the differences between the model's predicted and actual values. SSE is used to quantify the accuracy of a model, with a lower SSE indicating a better fit.



Q1.4)

Overfitting is when the model is overtrained/overly complex and fits the data too closely, so that noise and outliers are treated as actual patterns to follow. Underfitting happens when the model has not been trained long enough or is too simple to effectively capture the pattern. Underfitting causes the model to miss patterns in the data and has poor overall performance. 


Q1.5)

Splitting the data into a training and testing set allows us to train models on one of the split datasets and check the performance of the model on the other half of the data to check how well the model works. Testing different k values and comparing the SSE values helps identify which k values gives the best performance. This makes it easier to choose a model that works well across different data rather than just fitting the training data too closely.



Q1.6)

A class label prediction is definite, and each observation is categorized into a specific bucket that the model thinks the observation matches the best. The strength of class labels is that it is simple. The decision is made directly from the model and can be evaluated easily with accuracy. A weakness of class labels is that we can not tell the confidence of an observation, in that two observations with the same class labels could have vastly different probabilities. For example, one could have 51%, and the other could have 99%, yet both have the same label. A probability distribution assigns probabilities for each class rather than a definite label. The strength of this approach is that it better shows the classification confidence for each observation. For example, if an observation is 90% class 1 and 10% class 2, we can be more confident in the prediction than if the prediction was 55% class 1 and 45% class 2. A weakness is that the probability distributions are harder to interpret and could be misleading if the model is poorly trained.



In [11]:
# Your Phase 1 solution to the problem goes here.
#Q2.1
import pandas as pd
df = pd.read_csv('./data/land_mines.csv')
print(df.head(), "\n")
print(df.info(), "\n")
print(df.describe(), "\n")

print("number of missing values: \n")
print(df.isnull().sum(), "\n")

print(df['voltage'].value_counts(), "\n")
print(df['height'].value_counts(), "\n")
print(df['soil'].value_counts(), "\n")
print(df['mine_type'].value_counts(), "\n")


    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1 

<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None 

          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.00

Q2.1)

The dataset has 338 observations in total with three key features in votlage, height, and soil with the target label being mine_type. The dataset appears to have no missing values. The features are all values between 0 and 1. There arer 5 types of mines, with each type of mine generally appearing equally often. The mine types for each category range from 65 to 71, so no class appears significantly more than any other. Overall, the data is clean and is balanced. Therefore, the data is suitable for training a classification model.

In [15]:
#Q2.2
from sklearn.model_selection import train_test_split

x = df[['voltage', 'height', 'soil']]
y = df['mine_type']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.5, random_state=1, stratify=y)


In [16]:
#Q2.3
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

for k in range(1, 50):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(x_train, y_train)
    y_pred = knn.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(k, accuracy)

best_model = KNeighborsClassifier(n_neighbors=2)
best_model.fit(x_train, y_train)

1 0.3254437869822485
2 0.40236686390532544
3 0.3905325443786982
4 0.33727810650887574
5 0.3431952662721893
6 0.3136094674556213
7 0.3254437869822485
8 0.3076923076923077
9 0.31952662721893493
10 0.35502958579881655
11 0.33136094674556216
12 0.3076923076923077
13 0.3431952662721893
14 0.30177514792899407
15 0.3076923076923077
16 0.31952662721893493
17 0.33136094674556216
18 0.33136094674556216
19 0.30177514792899407
20 0.28994082840236685
21 0.28402366863905326
22 0.31952662721893493
23 0.2958579881656805
24 0.3076923076923077
25 0.28994082840236685
26 0.2958579881656805
27 0.3076923076923077
28 0.30177514792899407
29 0.3136094674556213
30 0.30177514792899407
31 0.31952662721893493
32 0.33727810650887574
33 0.34911242603550297
34 0.3254437869822485
35 0.3136094674556213
36 0.3136094674556213
37 0.33136094674556216
38 0.3254437869822485
39 0.35502958579881655
40 0.3076923076923077
41 0.3076923076923077
42 0.2958579881656805
43 0.3136094674556213
44 0.3254437869822485
45 0.319526627218934

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",2
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](5,)","[1,2,3,4,5]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


Q2.3)

I tested different values of k and compared the classification accuracy for the respective k values. I chose k=2 for the knn classifier because k=2 resulted in the highest accuracy of 0.402 for k values between 1 and 49.

In [18]:
#Q2.4

from sklearn.metrics import confusion_matrix
c = confusion_matrix(y_test, y_pred = best_model.predict(x_test))
print(c, "\n")
print(accuracy_score(y_test, y_pred), "\n")

[[18  1 12  3  1]
 [ 0 29  1  5  0]
 [ 7  1 16  8  1]
 [15  5 10  3  0]
 [14  1 14  2  2]] 

0.3431952662721893 



Q2.4)

The knn model has an accuracy of around 0.343. The most accurate is type 2 where there are 29 correct classifications. Type 1 and 3 are slightly less accurate with 18 and 16 correct classifications correctly. They also sometimes missclasify as the other type, with 12 type 1 misclassified as type 3 and 7 type 3 misclassified as type 1. Type 4 and 5 are the least accurate with 3 and 2 correct classifications. Type 4 and 5 also often misclassify as Type 1 or Type 3.

Q2.5) 

The model should mainly be used as an additional tool that can help with identfying a mine, in this instance, rather than the only method of classification. The model has a classification accuracy of aroudn 34.3%, which is relatively low and often misclassifies mine types as shown with mine type 4 and 5. Becasue of this, the model's classification should be used with some other classification method to verify the predictions before making a final decision.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [5]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]